# M11.1 — Export M8 Step 3 Fourier → Hub model clone

Plan: [`plans/milestone_11/11_huggingface_artifacts_plan.md`](../../plans/milestone_11/11_huggingface_artifacts_plan.md) §B.  
Architecture: [`plans/00_architecture.md`](../../plans/00_architecture.md) §5.7.

Extracts **Fourier only** from `checkpoints/m8/m08_train_val_test_xyz.pt` into the local clone of
[`tbhugging/singleview_cnn_fourier`](https://huggingface.co/tbhugging/singleview_cnn_fourier).

Machine-local staging path: gitignored `configs/hf/local.toml`
(copy from `configs/hf/local.toml.example`). This notebook writes weights + card
into that clone; it does **not** push to the Hub.


In [1]:
from pathlib import Path
import shutil
import sys
import subprocess

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

for name in ("build", "dist"):
    shutil.rmtree(ROOT / name, ignore_errors=True)
for egg in (ROOT / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        f"{ROOT}[dl,hf,dev]",
        "-c",
        str(ROOT / "requirements.txt"),
    ]
)

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")


ROOT=.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Resolve local clone + export

Requires:
- `configs/hf/local.toml` with `models.singleview_cnn_fourier.local_clone`
- Study checkpoint `checkpoints/m8/m08_train_val_test_xyz.pt`


In [2]:
from IPython.display import Markdown, display

from tomography_ml_validation.milestone_11 import (
    export_singleview_cnn_fourier,
    resolve_singleview_cnn_fourier_paths,
)

paths = resolve_singleview_cnn_fourier_paths(ROOT)
display(Markdown(
    f"**Hub:** [`{paths.hub_id}`]({paths.hub_url})  \n"
    f"**Local clone:** `{paths.local_clone}`"
))

result = export_singleview_cnn_fourier(ROOT)
display(Markdown(
    f"Wrote `{display_path(result.weights_path)}`, "
    f"`{display_path(result.config_path)}`, "
    f"`{display_path(result.readme_path)}`  \n"
    f"n_params={result.n_params}  lr={result.lr:g}  "
    f"metrics={{{', '.join(f'{k}={v:.4f}' for k, v in sorted(result.metrics.items()))}}}"
))


**Hub:** [`tbhugging/singleview_cnn_fourier`](https://huggingface.co/tbhugging/singleview_cnn_fourier)  
**Local clone:** `/Users/thomasbraschler/huggingface/singleview_cnn_fourier`

Wrote `~/huggingface/singleview_cnn_fourier/pytorch_model.bin`, `~/huggingface/singleview_cnn_fourier/config.json`, `~/huggingface/singleview_cnn_fourier/README.md`  
n_params=31811  lr=0.03  metrics={lr=0.0300, n_params=31811.0000, test_RMSE_total=1.4912, validation_RMSE_total=2.2755}

## Smoke-check reload


In [3]:
import json
import torch

from tomography_ml.localization.builders import materialize_lazy_modules
from tomography_ml.studies.single_view_m8 import make_m8_single_view_model

cfg = json.loads(result.config_path.read_text(encoding="utf-8"))
assert cfg["architecture"] == "fourier"
assert cfg["x_field"] == "anomaly_ref"

model = make_m8_single_view_model("fourier", n_outputs=3, device="cpu")
materialize_lazy_modules(model, torch.zeros(1, 1, cfg["image_height"], cfg["image_width"]))
state = torch.load(result.weights_path, map_location="cpu", weights_only=True)
model.load_state_dict(state, strict=True)
model.eval()
xyz = model(torch.zeros(1, 1, cfg["image_height"], cfg["image_width"]))
print("reload ok", tuple(xyz.shape), "config n_params=", cfg["n_params"])


reload ok (1, 3) config n_params= 31811


## Hub upload (manual)

After reviewing the local clone:

```bash
hf auth login
cd "$local_clone"   # from configs/hf/local.toml
hf upload tbhugging/singleview_cnn_fourier . .
# or: git add -A && git commit && git push
```
